# Swing-Up v3 — Curriculum SAC with Energy Shaping

Four-phase curriculum that progressively widens the initial-angle distribution, teaching stabilisation before swing-up.

| | v1 (two-PPO) | v2 (single SAC, hanging) | v3 (this notebook) |
|---|---|---|---|
| **Agents** | swing-up PPO + stabiliser PPO | single SAC | single SAC, curriculum |
| **Init** | hanging | hanging | ±15° → ±60° → ±120° → ±180° |
| **Reward** | alive − distance | height + balance bonus | height + balance bonus + energy shaping (phases 3–4) |
| **Seam** | PPO→PPO handoff (failure point) | none | none |

**Why curriculum?**  Starting every episode from hanging, the upright attractor occupies a tiny fraction of visited states — the agent rarely encounters it. Starting near-upright teaches stabilisation first; widening the range in phases bridges to the full task without restarting from scratch.

**Why energy shaping in phases 3–4?**  When θ₁ > 60° the height reward is nearly flat — all far-from-upright states look equally bad. Adding `−λ(E − E★)²` rewards the agent for reaching the *energy level* of the upright equilibrium, guiding energy-pumping behaviour even when the height improvement per step is small. The term is zero at the upright equilibrium so it does not interfere with fine balance.

**Balance gate** (unchanged): `|θ₁| < 0.10 rad AND |θ₂| < 0.10 rad AND |θ̇₁| < 0.50 rad/s AND |θ̇₂| < 0.50 rad/s` — velocity component prevents bonus farming via fast passes through upright.

In [6]:
import sys
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gymnasium as gym
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

CWD = pathlib.Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [CWD, *CWD.parents] if (p / "pyproject.toml").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Environment ───────────────────────────────────────────────────────────────
ENV_ID    = "InvertedDoublePendulum-v5"
MODEL_DIR = PROJECT_ROOT / "data" / "sac_curriculum"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Balance gate ──────────────────────────────────────────────────────────────
ANGLE_THRESH  = 0.10    # rad  — ~6°
VEL_THRESH    = 0.50    # rad/s
BALANCE_BONUS = 3.0     # reward inside gate
BALANCE_STEPS = 200     # consecutive steps inside gate → success  (10 s)
CART_LIMIT    = 2.4     # m — cart termination
ANGLE_NOISE   = 0.05    # rad — init jitter

# ── Energy shaping (phases 3–4 only) ─────────────────────────────────────────
# r_energy = −ENERGY_WEIGHT × (E_proxy − E_TARGET)²
# E_proxy  = cos θ₁ + cos(θ₁+θ₂) + 0.5(θ̇₁² + θ̇₂²)
# E_TARGET = 2.0  → upright equilibrium, zero velocity
ENERGY_WEIGHT = 0.10
E_TARGET      = 2.0

# ── Curriculum ────────────────────────────────────────────────────────────────
CURRICULUM_PHASES = [
    {"name": "Phase 1 — stabilise",    "angle_range": np.radians(15),  "steps": 400_000, "energy_shaping": False},
    {"name": "Phase 2 — medium swing", "angle_range": np.radians(60),  "steps": 500_000, "energy_shaping": False},
    {"name": "Phase 3 — large swing",  "angle_range": np.radians(120), "steps": 550_000, "energy_shaping": True},
    {"name": "Phase 4 — full hang",    "angle_range": np.radians(180), "steps": 550_000, "energy_shaping": True},
]
TOTAL_STEPS = sum(p["steps"] for p in CURRICULUM_PHASES)

# ── Training ──────────────────────────────────────────────────────────────────
N_ENVS         = 8     # envs stepped per tick  (DummyVecEnv — data diversity, not wall-clock)
GRADIENT_STEPS = 4     # 4 updates per 8 env steps → 1:2 update-to-data ratio
TRAIN_EP_STEPS = 1_000 # short training episodes → faster resets, more diverse buffer
MAX_EP_STEPS   = 5_000 # evaluation cap (250 s)
N_EVAL         = 20

_env    = gym.make(ENV_ID)
DT      = _env.unwrapped.dt
OBS_DIM = _env.observation_space.shape[0]
ACT_DIM = _env.action_space.shape[0]
_env.close()

print(f"DT={DT}s  OBS={OBS_DIM}  ACT={ACT_DIM}")
print(f"Balance gate : |θ| < {np.degrees(ANGLE_THRESH):.0f}°  |θ̇| < {VEL_THRESH} rad/s")
print(f"Success      : {BALANCE_STEPS} consecutive steps ({BALANCE_STEPS*DT:.0f} s)")
print(f"\nCurriculum phases  (total {TOTAL_STEPS:,} steps):")
for i, p in enumerate(CURRICULUM_PHASES):
    es = "  + energy shaping" if p["energy_shaping"] else ""
    print(f"  {i+1}. {p['name']:<28}  ±{np.degrees(p['angle_range']):.0f}°{es}  ({p['steps']//1000}k steps)")

DT=0.05s  OBS=9  ACT=1
Balance gate : |θ| < 6°  |θ̇| < 0.5 rad/s
Success      : 200 consecutive steps (10 s)

Curriculum phases  (total 2,000,000 steps):
  1. Phase 1 — stabilise           ±15°  (400k steps)
  2. Phase 2 — medium swing        ±60°  (500k steps)
  3. Phase 3 — large swing         ±120°  + energy shaping  (550k steps)
  4. Phase 4 — full hang           ±180°  + energy shaping  (550k steps)


In [7]:
def obs_to_state6(obs: np.ndarray) -> np.ndarray:
    """9-dim MuJoCo obs → [x, θ₁, θ₂, ẋ, θ̇₁, θ̇₂]."""
    return np.array([
        obs[0],
        np.arctan2(obs[1], obs[3]),
        np.arctan2(obs[2], obs[4]),
        obs[5], obs[6], obs[7],
    ], dtype=np.float64)


def near_balanced(obs: np.ndarray) -> bool:
    """True when both poles are near vertical AND angular velocities are small."""
    th1 = np.arctan2(obs[1], obs[3])
    th2 = np.arctan2(obs[2], obs[4])
    return (
        abs(th1) < ANGLE_THRESH and abs(th2) < ANGLE_THRESH
        and abs(obs[6]) < VEL_THRESH and abs(obs[7]) < VEL_THRESH
    )


def energy_proxy(obs: np.ndarray) -> float:
    """
    Proxy for total mechanical energy, normalised so E★ = 2.0 at the upright equilibrium.

    PE proxy  cos θ₁ + cos(θ₁+θ₂)   ∈ [−2, +2]
    KE proxy  0.5(θ̇₁² + θ̇₂²)
    At hanging (θ₁=π, velocities=0): E ≈ −2   →  error = −4
    At upright (θ₁=0, velocities=0): E ≈ +2   →  error =  0
    """
    h  = obs[3] + obs[3] * obs[4] - obs[1] * obs[2]   # cos θ₁ + cos(θ₁+θ₂)
    ke = 0.5 * (obs[6] ** 2 + obs[7] ** 2)
    return float(h + ke)


def height_reward(obs: np.ndarray) -> float:
    h = obs[3] + obs[3] * obs[4] - obs[1] * obs[2]
    return float(h - 0.01 * obs[0] ** 2)

---

## Environment Wrappers

Two wrappers, one per role:

### `SwingUpEvalWrapper`  (evaluation only)
Always initialises from hanging (θ₁ ≈ π). Uses only height + balance-bonus reward — **no energy shaping** — so evaluation results are comparable across experiments and reflect the real task.

### `CurriculumWrapper`  (training only)
Samples θ₁ uniformly from `[−angle_range, +angle_range]` (radians from upright). For phases 3–4 it also adds the energy-shaping term to the reward.

**Reward** (both wrappers before energy shaping):
```
r = cos θ₁ + cos(θ₁+θ₂) − 0.01 x²   ← height proxy, ∈ [−2, +2]
  + BALANCE_BONUS                      ← 3.0 if near_balanced(obs)
```

**Energy-shaping term** (`CurriculumWrapper`, phases 3–4):
```
r -= ENERGY_WEIGHT × (E_proxy − E★)²
```
`E_proxy = cos θ₁ + cos(θ₁+θ₂) + 0.5(θ̇₁² + θ̇₂²)`,  `E★ = 2.0`

In [8]:
class SwingUpEvalWrapper(gym.Wrapper):
    """Evaluation wrapper — hanging start, no energy shaping."""

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        env  = self.unwrapped
        qpos = env.data.qpos.copy()
        qpos[0] = 0.0
        qpos[1] = np.pi + env.np_random.uniform(-ANGLE_NOISE, ANGLE_NOISE)
        qpos[2] = env.np_random.uniform(-ANGLE_NOISE, ANGLE_NOISE)
        env.set_state(qpos, np.zeros(env.model.nv))
        return env._get_obs(), info

    def step(self, action):
        obs, _, _, truncated, info = self.env.step(action)
        reward     = height_reward(obs) + (BALANCE_BONUS if near_balanced(obs) else 0.0)
        terminated = bool(abs(obs[0]) > CART_LIMIT)
        return obs, reward, terminated, truncated, info


# Sanity check
_e = SwingUpEvalWrapper(gym.make(ENV_ID))
obs_hang, _ = _e.reset(seed=0)
print(f"Height reward at hanging : {height_reward(obs_hang):.2f}  (expected −2.00)")
print(f"Energy proxy  at hanging : {energy_proxy(obs_hang):.2f}  (E★={E_TARGET}  err={energy_proxy(obs_hang)-E_TARGET:.2f})")
print(f"near_balanced at hanging : {near_balanced(obs_hang)}")
_e.close()

Height reward at hanging : -2.00  (expected −2.00)
Energy proxy  at hanging : -2.00  (E★=2.0  err=-4.00)
near_balanced at hanging : False


In [9]:
class CurriculumWrapper(gym.Wrapper):
    """
    Training wrapper — samples θ₁ from ±angle_range around upright.
    Adds energy-shaping term when phase_config['energy_shaping'] is True.
    """

    def __init__(self, env, phase_config: dict):
        super().__init__(env)
        self.cfg = phase_config

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        env  = self.unwrapped
        qpos = env.data.qpos.copy()
        qpos[0] = 0.0
        qpos[1] = env.np_random.uniform(-self.cfg["angle_range"], self.cfg["angle_range"])
        qpos[2] = env.np_random.uniform(-ANGLE_NOISE, ANGLE_NOISE)
        env.set_state(qpos, np.zeros(env.model.nv))
        return env._get_obs(), info

    def step(self, action):
        obs, _, _, truncated, info = self.env.step(action)
        reward     = height_reward(obs) + (BALANCE_BONUS if near_balanced(obs) else 0.0)
        if self.cfg["energy_shaping"]:
            e_err   = energy_proxy(obs) - E_TARGET
            reward -= ENERGY_WEIGHT * e_err ** 2
        terminated = bool(abs(obs[0]) > CART_LIMIT)
        return obs, reward, terminated, truncated, info


# Verify energy penalty at hanging start (should be strongly negative)
_ec = CurriculumWrapper(gym.make(ENV_ID), CURRICULUM_PHASES[3])  # phase 4
obs_p4, _ = _ec.reset(seed=0)
e_p4 = energy_proxy(obs_p4)
print(f"Phase 4 init (≈hanging):")
print(f"  energy proxy   : {e_p4:.2f}  (E★={E_TARGET})")
print(f"  energy penalty : {-ENERGY_WEIGHT*(e_p4-E_TARGET)**2:.3f}  (ENERGY_WEIGHT={ENERGY_WEIGHT})")
_ec.close()

Phase 4 init (≈hanging):
  energy proxy   : 1.55  (E★=2.0)
  energy penalty : -0.020  (ENERGY_WEIGHT=0.1)


---

## Curriculum Training

A single SAC agent (`[256, 256]` MLP, `ent_coef='auto'`) is trained across four phases. The **replay buffer accumulates across phases** — transitions from near-upright phases remain in the buffer during later full-hang phases, providing stable gradient signal throughout.

| Phase | Init θ₁ range | Energy shaping | Steps |
|---|---|---|---|
| 1 — stabilise    | ±15°  | no  | 400k |
| 2 — medium swing | ±60°  | no  | 500k |
| 3 — large swing  | ±120° | yes | 550k |
| 4 — full hang    | ±180° | yes | 550k |

A fresh `train_env` is created for each phase (different reset distribution); `model.set_env()` hot-swaps it without resetting the replay buffer. `EvalCallback` evaluates from the **phase's own distribution** — this gives informative reward signal throughout, not only once the agent can swing up from hanging.

`reset_num_timesteps=False` from phase 2 onward keeps the learning-rate schedule and step counter continuous. Phase checkpoints allow resuming if interrupted.

In [10]:
FINAL_MODEL = MODEL_DIR / "curriculum_final.zip"


class TqdmCallback(BaseCallback):
    """Single tqdm bar per phase — updates in-place in Jupyter (no rich multi-bar flicker)."""
    def __init__(self, total_steps: int, desc: str = ""):
        super().__init__()
        self._total = total_steps
        self._desc  = desc
        self._pbar  = None
        self._start = 0   # global step offset at phase start

    def _on_training_start(self):
        self._start = self.num_timesteps   # capture offset so phase bar starts at 0
        self._pbar = tqdm(total=self._total, desc=self._desc, unit="step", leave=True)

    def _on_step(self) -> bool:
        self._pbar.n = self.num_timesteps - self._start
        self._pbar.refresh()
        return True

    def _on_training_end(self):
        self._pbar.n = self._total
        self._pbar.refresh()
        self._pbar.close()


def _make_train_env(phase):
    return make_vec_env(
        lambda p=phase: Monitor(CurriculumWrapper(
            gym.make(ENV_ID, max_episode_steps=TRAIN_EP_STEPS), p)),
        n_envs=N_ENVS,
    )


def _make_phase_eval_env(phase):
    return Monitor(CurriculumWrapper(
        gym.make(ENV_ID, max_episode_steps=MAX_EP_STEPS), phase))


if FINAL_MODEL.exists():
    print(f"Loading trained model from {FINAL_MODEL}")
    model = SAC.load(str(FINAL_MODEL))

else:
    # Initialise SAC on phase 1 env to set obs/action dimensions
    _bootstrap_env = _make_train_env(CURRICULUM_PHASES[0])
    model = SAC(
        "MlpPolicy", _bootstrap_env,
        learning_rate=3e-4,
        buffer_size=1_000_000,
        batch_size=256,
        tau=0.005,
        gamma=0.99,
        ent_coef="auto",
        gradient_steps=GRADIENT_STEPS,
        policy_kwargs=dict(net_arch=[256, 256]),
        verbose=0,
        tensorboard_log=str(MODEL_DIR / "tb"),
    )
    _bootstrap_env.close()

    # ── Phase loop ─────────────────────────────────────────────────────────
    for i, phase in enumerate(CURRICULUM_PHASES):
        phase_ckpt = MODEL_DIR / f"phase{i+1}_best" / "best_model.zip"

        if phase_ckpt.exists():
            print(f"Phase {i+1}: skipping — checkpoint found at {phase_ckpt.parent}")
            model = SAC.load(str(phase_ckpt))
            continue

        label = (f"{phase['name']}  ±{np.degrees(phase['angle_range']):.0f}°"
                 f"{'  + energy shaping' if phase['energy_shaping'] else ''}")
        print(f"\n{'─'*60}\n{label}  ({phase['steps']//1000}k steps)")

        train_env = _make_train_env(phase)
        eval_env  = _make_phase_eval_env(phase)
        model.set_env(train_env)

        eval_cb = EvalCallback(
            eval_env,
            best_model_save_path=str(MODEL_DIR / f"phase{i+1}_best"),
            log_path=str(MODEL_DIR / f"phase{i+1}_log"),
            eval_freq=max(25_000 // N_ENVS, 1),
            n_eval_episodes=10,
            deterministic=True,
            verbose=0,
        )
        tqdm_cb = TqdmCallback(phase["steps"], desc=f"P{i+1}")

        model.learn(
            total_timesteps=phase["steps"],
            callback=[eval_cb, tqdm_cb],
            progress_bar=False,
            reset_num_timesteps=(i == 0),
        )
        train_env.close()
        eval_env.close()

    model.save(str(MODEL_DIR / "curriculum_final"))
    print("\nAll phases complete. Final model saved.")



────────────────────────────────────────────────────────────
Phase 1 — stabilise  ±15°  (400k steps)


P1:   0%|          | 0/400000 [00:00<?, ?step/s]


────────────────────────────────────────────────────────────
Phase 2 — medium swing  ±60°  (500k steps)


P2:   0%|          | 0/500000 [00:00<?, ?step/s]

KeyboardInterrupt: 

---

## Evaluation

All evaluation uses `SwingUpEvalWrapper`: **hanging start, no energy-shaping reward**. This reflects the real task and is directly comparable to v1/v2 results regardless of how the agent was trained.

Each episode terminates on:
- **Success**: `|θ₁| < 0.10 rad AND |θ₂| < 0.10 rad AND |θ̇₁,₂| < 0.50 rad/s` for 200 consecutive steps (10 s)
- **Cart OOB**: `|x| > 2.4 m`
- **Timeout**: 250 s elapsed

The `N_EVAL` episodes are **run in parallel** via `ThreadPoolExecutor`. Each thread creates its own environment, so there is no shared mutable state. MuJoCo's C simulation releases the GIL, so the threads run concurrently.

In [ ]:
def run_episode(model, seed: int = 0):
    """Roll out one episode from hanging using SwingUpEvalWrapper.

    Returns (states, actions, rewards, success, diagnostics).
    Thread-safe: creates its own env instance.
    """
    env = SwingUpEvalWrapper(gym.make(ENV_ID, max_episode_steps=MAX_EP_STEPS))
    obs, _ = env.reset(seed=seed)

    states, actions, rewards = [], [], []
    consecutive_balanced = 0
    max_consecutive      = 0
    first_entry_step     = None
    success              = False
    termination          = "timeout"

    for step in range(MAX_EP_STEPS):
        action, _ = model.predict(obs, deterministic=True)
        states.append(obs_to_state6(obs))
        actions.append(float(action[0]))
        obs, r, terminated, truncated, _ = env.step(action)
        rewards.append(r)

        if near_balanced(obs):
            if first_entry_step is None:
                first_entry_step = step
            consecutive_balanced += 1
            max_consecutive = max(max_consecutive, consecutive_balanced)
            if consecutive_balanced >= BALANCE_STEPS:
                success     = True
                termination = "success"
                break
        else:
            consecutive_balanced = 0

        if terminated:
            termination = "cart_oob"
            break
        if truncated:
            break

    env.close()
    diag = {
        "max_consecutive":   max_consecutive,
        "first_entry_step":  first_entry_step,
        "first_entry_time":  None if first_entry_step is None else first_entry_step * DT,
        "termination":       termination,
        "max_height_reward": float(max(
            (r - BALANCE_BONUS if r > 2.0 else r) for r in rewards
        )) if rewards else -2.0,
    }
    return np.array(states), np.array(actions), np.array(rewards), success, diag

In [ ]:
print(f"Evaluating {N_EVAL} episodes in parallel ({min(N_EVAL, N_ENVS)} threads) ...")
with ThreadPoolExecutor(max_workers=min(N_EVAL, N_ENVS)) as pool:
    results = list(pool.map(lambda s: run_episode(model, seed=s), range(N_EVAL)))

ep_lens     = [len(r[0]) for r in results]
successes   = [r[3]      for r in results]
diags       = [r[4]      for r in results]

n_success  = sum(successes)
n_reached  = sum(1 for d in diags if d["first_entry_step"] is not None)
n_cart_oob = sum(1 for d in diags if d["termination"] == "cart_oob")
n_timeout  = sum(1 for d in diags if d["termination"] == "timeout")
entry_times = [d["first_entry_time"] for d in diags if d["first_entry_time"] is not None]
max_consecs = [d["max_consecutive"] for d in diags]
best_seed   = next((i for i in range(N_EVAL) if successes[i]), 0)

print(f"\n── Results ──────────────────────────────────────────────")
print(f"  Success (held {BALANCE_STEPS} steps) : {n_success}/{N_EVAL}  ({100*n_success/N_EVAL:.0f}%)")
print(f"  Reached gate at all      : {n_reached}/{N_EVAL}")
if entry_times:
    print(f"  Mean time to first entry : {np.mean(entry_times):.1f}s  "
          f"(range {min(entry_times):.1f}–{max(entry_times):.1f}s)")
print(f"  Max consecutive balanced : {max(max_consecs)} steps  (target {BALANCE_STEPS})")
print(f"  Median max consecutive   : {int(np.median(max_consecs))} steps")
print(f"\n── Failure breakdown ─────────────────────────────────────")
print(f"  Success  : {n_success}")
print(f"  Timeout  : {n_timeout}")
print(f"  Cart OOB : {n_cart_oob}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 3))

axes[0].bar(
    np.arange(N_EVAL), [l * DT for l in ep_lens],
    color=["seagreen" if s else "steelblue" for s in successes],
    alpha=0.75, width=0.8,
)
axes[0].set_xlabel("Episode seed")
axes[0].set_ylabel("Time (s)")
axes[0].set_title(f"Episode duration  ({n_success}/{N_EVAL} success)")
axes[0].legend(handles=[
    mpatches.Patch(color="seagreen",  alpha=0.7, label="success"),
    mpatches.Patch(color="steelblue", alpha=0.7, label="failed"),
], fontsize=9)

colors = ["seagreen" if s else ("tomato" if d["termination"] == "cart_oob" else "steelblue")
          for s, d in zip(successes, diags)]
axes[1].bar(np.arange(N_EVAL), max_consecs, color=colors, alpha=0.75, width=0.8)
axes[1].axhline(BALANCE_STEPS, ls="--", color="black", lw=1, label=f"target ({BALANCE_STEPS})")
axes[1].set_xlabel("Episode seed")
axes[1].set_ylabel("Steps")
axes[1].set_title("Max consecutive steps inside balance gate")
axes[1].legend(handles=[
    mpatches.Patch(color="seagreen", alpha=0.7, label="success"),
    mpatches.Patch(color="steelblue", alpha=0.7, label="timeout"),
    mpatches.Patch(color="tomato",    alpha=0.7, label="cart OOB"),
], fontsize=9)

entry_by_seed = [d["first_entry_time"] if d["first_entry_time"] is not None else np.nan
                 for d in diags]
axes[2].bar(np.arange(N_EVAL), entry_by_seed,
            color=["seagreen" if s else "steelblue" for s in successes],
            alpha=0.75, width=0.8)
axes[2].set_xlabel("Episode seed")
axes[2].set_ylabel("Time (s)")
axes[2].set_title("Time to first gate entry  (nan = never reached)")

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, Markdown

def _pct(n, total=N_EVAL):
    return f"{n}/{total} ({100*n/total:.0f}%)"

ok_ep_lens  = [ep_lens[i] * DT for i in range(N_EVAL) if successes[i]]
mean_entry  = f"{np.mean(entry_times):.1f} s" if entry_times else "—"
entry_range = (f"{min(entry_times):.1f}–{max(entry_times):.1f} s"
               if len(entry_times) > 1 else mean_entry)
mean_dur    = f"{np.mean(ok_ep_lens):.1f} s" if ok_ep_lens else "—"

rate = n_success / N_EVAL
if rate >= 0.80:
    assessment = "**Strong** — curriculum + energy shaping solved the task."
elif rate >= 0.50:
    assessment = "**Moderate** — majority succeed; further tuning may push above 80%."
elif n_reached / N_EVAL >= 0.50:
    assessment = ("**Partial** — agent reaches the gate consistently but cannot hold it. "
                  "Consider increasing `BALANCE_STEPS` budget or `BALANCE_BONUS`.")
else:
    assessment = ("**Weak** — agent rarely reaches the gate. "
                  "Adjust `ENERGY_WEIGHT` or phase step budgets.")

display(Markdown(f"""
## Results Summary

Training: **{TOTAL_STEPS:,} steps** across {len(CURRICULUM_PHASES)} curriculum phases  ·  {N_ENVS} parallel envs

| Metric | Value |
|---|---|
| **Success rate** | {_pct(n_success)} |
| Reached balance gate | {_pct(n_reached)} |
| Mean time to first entry | {mean_entry} (range {entry_range}) |
| Mean episode duration (successes) | {mean_dur} |
| Max consecutive balanced steps | {max(max_consecs)} / {BALANCE_STEPS} |
| Median max consecutive | {int(np.median(max_consecs))} steps |
| Timeout failures | {n_timeout} |
| Cart OOB failures | {n_cart_oob} |

**Assessment**: {assessment}

---
*{N_EVAL} episodes · hanging start · gate: |θ| < {np.degrees(ANGLE_THRESH):.0f}° · |θ̇| < {VEL_THRESH} rad/s · hold {BALANCE_STEPS*DT:.0f} s*
"""))

In [ ]:
def plot_episode(states, actions, rewards, success, title="SAC swing-up"):
    t      = np.arange(len(states)) * DT
    status = "SUCCESS" if success else "FAILED"
    color  = "seagreen" if success else "crimson"

    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

    axes[0].plot(t, np.degrees(states[:, 1]), label="θ₁", color="steelblue")
    axes[0].plot(t, np.degrees(states[:, 2]), label="θ₂", color="darkorange")
    axes[0].axhline(0, ls="--", color="gray", lw=0.8)
    axes[0].axhline( np.degrees(ANGLE_THRESH), ls=":", color="seagreen", lw=0.8)
    axes[0].axhline(-np.degrees(ANGLE_THRESH), ls=":", color="seagreen", lw=0.8)
    axes[0].set_ylabel("Angle (deg)")
    axes[0].legend(fontsize=9)

    axes[1].plot(t, states[:, 0], color="steelblue")
    axes[1].axhline(0, ls="--", color="gray", lw=0.8)
    axes[1].set_ylabel("Cart x (m)")

    axes[2].plot(t, actions, color="crimson", lw=0.8)
    axes[2].axhline(0, ls="--", color="gray", lw=0.8)
    axes[2].set_ylabel("Action")

    axes[3].plot(t, rewards, color="goldenrod", lw=0.8)
    axes[3].axhline(2.0, ls=":", color="seagreen", lw=0.8, label="max height reward")
    axes[3].set_ylabel("Reward")
    axes[3].set_xlabel("Time (s)")
    axes[3].legend(fontsize=9)

    fig.suptitle(f"{title}  |  {len(states)*DT:.1f}s  [{status}]",
                 fontsize=11, color=color)
    plt.tight_layout()
    plt.show()


# Reuse trajectory already collected in the parallel eval above
states, actions, rewards, success, diag = results[best_seed]
print(f"Seed {best_seed}: {len(states)} steps ({len(states)*DT:.1f}s)  "
      f"[{'SUCCESS' if success else 'FAILED'}]  "
      f"max_consec={diag['max_consecutive']}  term={diag['termination']}")
plot_episode(states, actions, rewards, success, title="Curriculum SAC swing-up")

---

## Animation

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML


def render_episode(model, seed: int = 0, speed: int = 3) -> HTML:
    env = SwingUpEvalWrapper(gym.make(ENV_ID, render_mode="rgb_array",
                                      max_episode_steps=MAX_EP_STEPS))
    obs, _ = env.reset(seed=seed)
    frames = [env.render()]
    consecutive_balanced = 0
    success = False
    balance_marker = None

    for step in range(MAX_EP_STEPS):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(action)
        frames.append(env.render())

        if near_balanced(obs):
            consecutive_balanced += 1
            if consecutive_balanced >= BALANCE_STEPS:
                success = True
                if balance_marker is None:
                    balance_marker = step - BALANCE_STEPS + 1
                break
        else:
            consecutive_balanced = 0

        if terminated or truncated:
            break
    env.close()

    n_steps = len(frames) - 1
    status  = "SUCCESS" if success else "FAILED"
    print(f"{n_steps} steps ({n_steps*DT:.1f}s)  [{status}]")

    fig, ax = plt.subplots(figsize=(4, 6))
    ax.axis("off")
    im = ax.imshow(frames[0])

    def _update(i):
        t = i * speed
        is_balanced = (balance_marker is not None and t >= balance_marker)
        fig.patch.set_facecolor("honeydew" if is_balanced else "aliceblue")
        im.set_data(frames[min(t, len(frames) - 1)])
        ax.set_title(f"t={t*DT:.1f}s", fontsize=9,
                     color="seagreen" if is_balanced else "royalblue")
        return [im]

    ani = animation.FuncAnimation(
        fig, _update,
        frames=range(len(frames) // speed + 1),
        interval=50, blit=False,
    )
    plt.close(fig)
    return HTML(ani.to_jshtml())


render_episode(model, seed=best_seed, speed=3)

---

## Balance Gate Sensitivity

The model was trained with `ANGLE_THRESH=0.10 rad` and `VEL_THRESH=0.50 rad/s`. Here we re-score the same 20 pre-collected trajectories against stricter and looser gate definitions — the **policy is unchanged**.

This reveals whether the agent *genuinely* achieves tight balance (success rate holds at stricter thresholds) or barely scrapes the training gate (success rate collapses at stricter thresholds).

In [ ]:
ANGLE_THRESHOLDS = [0.05, 0.10, 0.15, 0.20]
VEL_THRESHOLDS   = [0.25, 0.50, 1.00]

# Reuse trajectories from parallel eval — no extra simulation needed
all_states = [r[0] for r in results]

sensitivity = np.zeros((len(ANGLE_THRESHOLDS), len(VEL_THRESHOLDS)))
for ai, at in enumerate(ANGLE_THRESHOLDS):
    for vi, vt in enumerate(VEL_THRESHOLDS):
        n_ok = 0
        for states in all_states:
            consec = 0
            for state in states:
                th1, th2, th1d, th2d = state[1], state[2], state[4], state[5]
                if abs(th1) < at and abs(th2) < at and abs(th1d) < vt and abs(th2d) < vt:
                    consec += 1
                    if consec >= BALANCE_STEPS:
                        n_ok += 1
                        break
                else:
                    consec = 0
        sensitivity[ai, vi] = 100 * n_ok / N_EVAL

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(sensitivity, aspect="auto", vmin=0, vmax=100, cmap="YlGn")
ax.set_xticks(range(len(VEL_THRESHOLDS)))
ax.set_yticks(range(len(ANGLE_THRESHOLDS)))
ax.set_xticklabels([f"{v} rad/s" for v in VEL_THRESHOLDS])
ax.set_yticklabels([f"{np.degrees(a):.0f}°" for a in ANGLE_THRESHOLDS])
ax.set_xlabel("vel_thresh")
ax.set_ylabel("angle_thresh")
ax.set_title("Success % at different evaluation thresholds  (same trained policy)")
for ai in range(len(ANGLE_THRESHOLDS)):
    for vi in range(len(VEL_THRESHOLDS)):
        ax.text(vi, ai, f"{sensitivity[ai, vi]:.0f}%",
                ha="center", va="center", fontsize=11)
plt.colorbar(im, ax=ax, label="Success %")
plt.tight_layout()
plt.show()